# 《实用Python编程》教学代码
## 第5章 数据智能 Pytorch深度学习

### 示例代码5.34 检查PyTorch是否已安装成功以及是否有GPU可用

In [6]:
import torch
print(torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

2.10.0
cpu


### 示例代码5.35 几种常见的创建Tensor对象的方式

In [7]:
import torch
import numpy as np
# 使用随机值或常量值创建指定形状的张量
shape = (2, 3)
zeros_tensor = torch.zeros(shape)
ones_tensor = torch.ones(shape)
rand_tensor = torch.rand(shape)
# 从Python列表创建
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)
# 从NumPy数组创建（共享底层内存）
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
# 基于现有Tensor对象创建，将继承输入对象的形状、数据类型等属性
x_ones = torch.ones_like(x_data) # 创建一个2x2的元素值均为1的Tensor
x_rand = torch.rand_like(x_data, dtype=torch.float) # 创建一个2x2的元素为随机浮点数的Tensor

### 示例代码5.36 查看Tensor对象的属性

In [8]:
x = torch.rand((3, 4))
print(x)
print(x.ndim, x.shape, x.dtype, x.device, x.requires_grad)

tensor([[0.5217, 0.6527, 0.0611, 0.6350],
        [0.9570, 0.6523, 0.2518, 0.6213],
        [0.7632, 0.6986, 0.5673, 0.2776]])
2 torch.Size([3, 4]) torch.float32 cpu False


### 示例代码5.37 Tensor算术操作

In [10]:
tensor = torch.ones((2, 2))
print(tensor)
print(tensor + 2) # 张量与标量相加
print(tensor - 1) # 张量与标量相减
print(tensor * 3) # 张量与标量相乘
print(tensor / 2) # 张量与标量相除

tensor([[1., 1.],
        [1., 1.]])
tensor([[3., 3.],
        [3., 3.]])
tensor([[0., 0.],
        [0., 0.]])
tensor([[3., 3.],
        [3., 3.]])
tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])


### 示例代码5.38 Tensor逐元素相乘与矩阵乘法

In [11]:
a = torch.tensor([[1, 0], [0, 1]])
b = torch.tensor([[1, 2], [3, 4]])
c = a * b # 等同于 torch.mul(a, b)
d = a @ b # 等同于 torch.matmul(a, b)
print(c)
print(d)

tensor([[1, 0],
        [0, 4]])
tensor([[1, 2],
        [3, 4]])


### 示例代码5.39 Tensor形变操作

In [12]:
a = torch.rand((4, 4))
b = a.reshape(-1, 8)
c = a.reshape(2, -1)
d = a.reshape(-1)
print(a.shape, b.shape, c.shape, d.shape)

torch.Size([4, 4]) torch.Size([2, 8]) torch.Size([2, 8]) torch.Size([16])


### 示例代码5.40 Tensor合并

In [13]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])
cat_0 = torch.cat([a, b], dim=0) # [1,2,3,4,5,6]
stack_0 = torch.stack([a, b], dim=0) # [[1,2,3], [4,5,6]]
stack_1 = torch.stack([a, b], dim=1) # [[1,4], [2,5], [3,6]]
print(cat_0.shape, stack_0.shape, stack_1.shape)

torch.Size([6]) torch.Size([2, 3]) torch.Size([3, 2])


### 示例代码5.41 Tensor自动求导

In [15]:
x = torch.tensor([2.0], requires_grad=True)
y = x**2 + 3*x + 1
y.backward() # 自动计算梯度
print(x.grad)

tensor([7.])


### 示例代码5.42 检查是否有GPU算力可用

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.rand((2, 2)).to(device) # 移动到GPU或CPU上来完成后续计算
print(x.device)

cpu


## 案例一：基于BoW特征的英文影评情感识别

### 示例代码5.43 自定义Dataset子类用于读取BoW特征

In [17]:
from torch.utils.data import Dataset

class FeatSet(Dataset):
    def __init__(self, feats, labels):
        self.feats = feats
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        dense = self.feats[idx].toarray().squeeze()
        return torch.tensor(dense, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

### 示例代码5.44 创建两个DataLoader对象，分别用于加载训练集和测试集

#### 数据预处理：复用5.2.1节的代码加载imdb训练集和测试集，并基于训练集拟合BoW特征提取器

In [25]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

train_file = './output/imdb_labelled_train.csv'
test_file = './output/imdb_labelled_test.csv'
df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
sents_train, sents_test = df_train['sentence'].values, df_test['sentence'].values
y_train, y_test = df_train['label'].values, df_test['label'].values
print(f'训练样本数量：{len(sents_train)}')  # 800
print(f'测试样本数量：{len(sents_test)}')  # 200

fe = CountVectorizer()
fe.fit(sents_train) # 使用训练数据构建BoW特征提取器
vob = fe.get_feature_names_out()
vocab_size = len(vob)
print(f'词表大小：{vocab_size}') # 词表大小：2647

训练样本数量：800
测试样本数量：200
词表大小：2647


In [22]:
from torch.utils.data import DataLoader

train_ds = FeatSet(fe.transform(sents_train), y_train)
test_ds = FeatSet(fe.transform(sents_test), [0]*len(sents_test)) #推理阶段无真实标签可用

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(len(train_loader), len(test_loader)) # 13 7

13 7


### 示例代码5.45 定义一个仅有一个线性层的网络类SimpleNet

In [23]:
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, vocab_size, num_class=2):
        super().__init__()
        self.net = nn.Linear(vocab_size, num_class)

    def forward(self, x):
        return self.net(x)

### 示例代码5.46 创建SimpleNet对象并设置损失函数、优化器、调度器等训练选项

In [35]:
from transformers import get_linear_schedule_with_warmup

model = SimpleNet(vocab_size)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
model.to(device) #模型 → GPU
criterion = nn.CrossEntropyLoss() #设定损失函数
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)
max_epoch = 300
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
print("可训练参数及参数量:")
print([(name, param.numel()) for name, param in model.named_parameters() if param.requires_grad])

Using device: cpu
可训练参数及参数量:
[('net.weight', 5294), ('net.bias', 2)]


### 示例代码5.47 训练SimpleNet

In [36]:
import time

t0 = time.perf_counter() # 记录开始时刻
for epoch in range(max_epoch):
    model.train()
    total_loss = 0
    for batch in train_loader:
        xb, yb = batch
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    total_loss /= len(train_loader) # 注意要在内循环之外
    if epoch % 20 == 0: # 每20轮次打印一次训练损失
        print(f"epoch {epoch:02d} : Loss = {total_loss:.4f}")
if device.type == "cuda":
    torch.cuda.synchronize(device)
total_time = time.perf_counter() - t0 # 训练结束计时
print(f"Training finished in {total_time/60:.2f} min")

epoch 00 : Loss = 0.6942
epoch 20 : Loss = 0.6440
epoch 40 : Loss = 0.6047
epoch 60 : Loss = 0.5717
epoch 80 : Loss = 0.5441
epoch 100 : Loss = 0.5204
epoch 120 : Loss = 0.5019
epoch 140 : Loss = 0.4870
epoch 160 : Loss = 0.4742
epoch 180 : Loss = 0.4611
epoch 200 : Loss = 0.4521
epoch 220 : Loss = 0.4448
epoch 240 : Loss = 0.4384
epoch 260 : Loss = 0.4337
epoch 280 : Loss = 0.4327
Training finished in 0.07 min


### 示例代码5.48 SimpleNet性能评测

In [37]:
from sklearn.metrics import classification_report

model.eval() #将模型设置为推理模式
all_preds = []
with torch.no_grad(): # 禁用梯度计算，提高推理效率
    for xb, _ in test_loader:
        xb = xb.to(device)
        output = model(xb)
        preds = torch.argmax(output, dim=1).cpu().numpy() # 每个样本取得分最高的类别作为预测结果
        all_preds.extend(preds)
# 性能评测
print(classification_report(y_test, all_preds, digits=3))

              precision    recall  f1-score   support

           0      0.832     0.848     0.840       105
           1      0.828     0.811     0.819        95

    accuracy                          0.830       200
   macro avg      0.830     0.829     0.829       200
weighted avg      0.830     0.830     0.830       200



### 示例代码5.49 用额外样本测试SimpleNet

In [38]:
input_texts = ['this is a good movie', 'the movie is horrible', 'a horrible movie', 'not a bad movie']
input_ds = FeatSet(fe.transform(input_texts), [0]*len(input_texts))
input_loader = DataLoader(input_ds, batch_size=16, shuffle=False)
model.eval()
all_scores = []
with torch.no_grad():
    for batch in input_loader:
        xb, yb = batch
        xb = xb.to(device)
        yb = yb.to(device)
        output = model(xb)
        scores = torch.softmax(output, dim=1).cpu().numpy()
        all_scores.append(scores)
scores_simplenet = np.vstack(all_scores)
print(scores_simplenet)

[[0.4928521  0.50714785]
 [0.5756278  0.42437217]
 [0.5802901  0.41970998]
 [0.6585362  0.3414638 ]]


## 案例二：基于BERT的英文影评情感识别

### 示例代码5.50 从HuggingFace镜像下载BERT模型

In [40]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="google-bert/bert-base-uncased", local_dir="bert-base-uncased", endpoint="https://hf-mirror.com")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

flax_model.msgpack:   0%|          | 0.00/438M [00:00<?, ?B/s]

weight.bin:   0%|          | 0.00/532M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

'/Users/xirong/workspace/py2026fall/bert-base-uncased'

### 示例代码5.51 从本地加载BERT模型及其分词器

In [42]:
from transformers import BertTokenizerFast, BertForSequenceClassification

local_model_path = './bert-base-uncased'
model = BertForSequenceClassification.from_pretrained(local_model_path, num_labels=2)
num_params = sum([param.numel() for name, param in model.named_parameters() if param.requires_grad])
print(f"可训练参数量: {num_params}")
print("分类层:", model.classifier)
tokenizer = BertTokenizerFast.from_pretrained(local_model_path)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


可训练参数量: 109483778
分类层: Linear(in_features=768, out_features=2, bias=True)


### 示例代码5.52 使用BERT分词器

In [43]:
text = ["a good movie", "a bad movie"]
encoded = tokenizer(text, add_special_tokens=True, max_length=8, truncation=True, padding='max_length', return_tensors='pt')
decoded = tokenizer.batch_decode(encoded['input_ids'])
print(encoded)
print(decoded)

{'input_ids': tensor([[ 101, 1037, 2204, 3185,  102,    0,    0,    0],
        [ 101, 1037, 2919, 3185,  102,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0]])}
['[CLS] a good movie [SEP] [PAD] [PAD] [PAD]', '[CLS] a bad movie [SEP] [PAD] [PAD] [PAD]']


### 示例代码5.53 定义面向BERT模型的数据集类BertDataset

In [44]:
class BertDataset(Dataset):
    def __init__(self, sents, labels, tokenizer, max_len=64):
        self.texts = sents
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoded.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

### 示例代码5.54 创建BertDataset和DataLoader对象，用于加载训练数据

In [46]:
train_ds = BertDataset(sents_train, y_train, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
print(f'Training data set: {len(train_ds)}')
print(f'Number of training batches: {len(train_loader)}')

Training data set: 800
Number of training batches: 25


### 示例代码5.55 为微调BERT设定训练选项

In [47]:
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
max_epoch = 10
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

### 示例代码5.56 微调BERT模型

In [49]:
from tqdm import tqdm

t0 = time.perf_counter()
for epoch in range(max_epoch):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")
if device.type == "cuda":
    torch.cuda.synchronize(device)
total_time = time.perf_counter() - t0
print(f"Training finished in {total_time/60:.2f} min")

Epoch 1: 100%|████████████████████████████████████████████████████████████████████████████████| 25/25 [00:32<00:00,  1.31s/it]


Epoch 1: Loss = 0.4748


Epoch 2: 100%|████████████████████████████████████████████████████████████████████████████████| 25/25 [00:31<00:00,  1.27s/it]


Epoch 2: Loss = 0.1491


Epoch 3: 100%|████████████████████████████████████████████████████████████████████████████████| 25/25 [00:31<00:00,  1.27s/it]


...

Epoch 8: Loss = 0.0024


Epoch 9: 100%|████████████████████████████████████████████████████████████████████████████████| 25/25 [00:31<00:00,  1.26s/it]


Epoch 9: Loss = 0.0015


Epoch 10: 100%|███████████████████████████████████████████████████████████████████████████████| 25/25 [00:33<00:00,  1.35s/it]

Epoch 10: Loss = 0.0014
Training finished in 5.37 min


### 示例代码5.57 BERT性能评测

In [50]:
test_ds = BertDataset(sents_test, [0]*len(sents_test), tokenizer) #推理阶段无真实标签可用
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
model.eval()
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
print(classification_report(y_test, all_preds, digits=3))

              precision    recall  f1-score   support

           0      0.917     0.952     0.935       105
           1      0.945     0.905     0.925        95

    accuracy                          0.930       200
   macro avg      0.931     0.929     0.930       200
weighted avg      0.931     0.930     0.930       200



### 示例代码5.58 用示例代码5.49的样本测试微调后的BERT模型

In [51]:
input_texts = ['this is a good movie', 'the movie is horrible', 'a horrible movie', 'not a bad movie']
input_ds = BertDataset(input_texts, [0]*len(input_texts), tokenizer)
input_loader = DataLoader(input_ds, batch_size=16, shuffle=False)
model.eval()
all_scores = []
with torch.no_grad():
    for batch in input_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        scores = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        all_scores.append(scores)
scores_bert = np.vstack(all_scores)
print(scores_bert)

[[5.1692297e-04 9.9948299e-01]
 [9.9925023e-01 7.4973638e-04]
 [9.9897802e-01 1.0219773e-03]
 [1.3519935e-03 9.9864799e-01]]


### 示例代码5.59 以DataFrame表格形式呈现不同模型给出的正向情感得分

In [52]:
df = pd.DataFrame({'review':input_texts, 'SimpleNet':scores_simplenet[:,1], 'BERT':scores_bert[:,1]})
df

,review,SimpleNet,BERT
0,this is a good movie,0.507148,0.999483
1,the movie is horrible,0.424372,0.000750
2,a horrible movie,0.419710,0.001022
3,not a bad movie,0.341464,0.998648


## 基座模型与图文嵌入技术

### 示例代码5.60 从HuggingFace镜像批量下载基座模型

In [53]:
from huggingface_hub import snapshot_download

endpoint = 'https://hf-mirror.com'
model_list = [("sentence-transformers/all-MiniLM-L6-v2", "fm/all-MiniLM-L6-v2"), 
              ("moka-ai/m3e-small", "fm/m3e-small"), 
              ("sentence-transformers/clip-ViT-B-32", "fm/clip-ViT-B-32")]
for repo_id, local_dir in model_list:
    snapshot_download(repo_id=repo_id, local_dir=local_dir, endpoint=endpoint)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

## 基于MiniLM模型的英文影评情感识别
### 示例代码5.61 从本地加载MiniLM模型

In [54]:
import sentence_transformers as sentrans

model = sentrans.SentenceTransformer("./fm/all-MiniLM-L6-v2")
# 输出模型的基本信息
p = sum(p.numel() for p in model.parameters())
vocab_size = model.tokenizer.vocab_size
max_len = model.get_max_seq_length()
#d = model.get_sentence_embedding_dimension()
d = model.get_embedding_dimension()
print("总参数量", f"{p/1e6:.1f} M")
print("词典大小", vocab_size)
print("嵌入维度", d)
print("输入文本最大长度", max_len)

总参数量 22.7 M
词典大小 30522
嵌入维度 384
输入文本最大长度 256


### 示例代码5.62 将影评情感类别（负面、正面）表示为嵌入特征

In [56]:
label_text = ['A negative movie review is a piece of writing that expresses dissatisfaction with a film. It typically points out perceived flaws such as a weak plot, poor acting, slow pacing, bad dialogue, or technical shortcomings, and it generally advises potential viewers against watching the movie.', 
              'A positive movie review is a piece of writing that expresses satisfaction with a film. It praises elements such as the story, acting, direction, cinematography, or emotional impact, and it encourages potential viewers to watch the movie.']
label_emb = model.encode(label_text)

### 示例代码5.63 将影评表示为嵌入特征

In [61]:
x_train = model.encode(sents_train) #ZSR不需要训练样本，故该语句在ZSR时可忽略
x_test = model.encode(sents_test)
print(f'训练集特征向量尺寸: {x_train.shape}')
print(f'测试集特征向量尺寸: {x_test.shape}')

训练集特征向量尺寸: (800, 384)
测试集特征向量尺寸: (200, 384)


### 示例代码5.64 基于MiniLM特征的零样本影评情感识别

In [62]:
scores = sentrans.util.cos_sim(x_test, label_emb) #计算影评特征与情感类别特征的余弦相似度
pred = np.argmax(scores, axis=1) #取相似度较大者作为识别结果
pred = pred.cpu().numpy() #从torch.Tensor转为numpy.ndarray
report = classification_report(y_test, pred, digits=3)
print(report)

              precision    recall  f1-score   support

           0      0.856     0.848     0.852       105
           1      0.833     0.842     0.838        95

    accuracy                          0.845       200
   macro avg      0.845     0.845     0.845       200
weighted avg      0.845     0.845     0.845       200



### 示例代码5.65 基于MiniLM特征的影评情感识别LRC训练与评估

In [64]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
clf.fit(x_train, y_train)
pred = clf.predict(x_test)
report = classification_report(y_test, pred, digits=3)
print(report)

              precision    recall  f1-score   support

           0      0.914     0.914     0.914       105
           1      0.905     0.905     0.905        95

    accuracy                          0.910       200
   macro avg      0.910     0.910     0.910       200
weighted avg      0.910     0.910     0.910       200



## 基于M3E模型的中文酒店评论情感识别
### 示例代码5.66 从本地加载M3E模型

In [65]:
model = sentrans.SentenceTransformer("./fm/m3e-small")
# 输出模型的基本信息
p = sum(p.numel() for p in model.parameters())
vocab_size = model.tokenizer.vocab_size
max_len = model.get_max_seq_length()
#d = model.get_sentence_embedding_dimension()
d = model.get_embedding_dimension()
print("总参数量", f"{p/1e6:.1f} M")
print("词典大小", vocab_size)
print("嵌入维度", d)
print("输入文本最大长度", max_len)

总参数量 24.0 M
词典大小 21128
嵌入维度 512
输入文本最大长度 512


### 示例代码5.67 将酒店评论表示为M3E嵌入特征

#### 数据预处理：复用5.2.2节的代码加载酒店评论训练集和测试集

In [70]:
train_file = './output/hotel_labelled_train.csv'
test_file = './output/hotel_labelled_test.csv'
df_train, df_test  = pd.read_csv(train_file), pd.read_csv(test_file)
sents_train, sents_test = df_train['review'].values, df_test['review'].values
y_train, y_test = df_train['label'].values, df_test['label'].values

print(f'训练样本数量：{len(sents_train)}')  # 6212
print(f'测试样本数量：{len(sents_test)}')  # 1553

训练样本数量：6212
测试样本数量：1553


In [72]:
x_train = model.encode(sents_train) #ZSR不需要训练样本，故该语句在ZSR时可忽略
x_test = model.encode(sents_test)

print(f'训练集特征向量尺寸: {x_train.shape}') # 6212*512
print(f'测试集特征向量尺寸: {x_test.shape}')  # 1553*512

训练集特征向量尺寸: (6212, 512)
测试集特征向量尺寸: (1553, 512)


### 示例代码5.68 基于M3E特征的零样本酒店评论情感识别

In [73]:
label_text = ['负面酒店评论是指顾客在入住体验后，对酒店服务、设施、卫生、环境等方面表达不满或失望的评论。这类评论通常包含对具体问题的描述，如房间脏乱、服务态度差、噪音扰民、设施故障、价格与体验不符等，旨在提醒其他潜在住客或促使酒店改进。', 
              '正面酒店评论是顾客在入住后对酒店表示满意或赞赏的评价。这类评论通常提及服务热情周到、房间整洁舒适、设施齐全先进、位置便利、性价比高或超出预期等优点，旨在推荐给其他潜在住客并肯定酒店的表现。']
label_emb = model.encode(label_text)
scores = sentrans.util.cos_sim(x_test, label_emb)
pred = np.argmax(scores, axis=1)
pred = pred.cpu().numpy()
report = classification_report(y_test, pred, digits=3)
print(report)

              precision    recall  f1-score   support

           0      0.595     0.941     0.729       476
           1      0.965     0.717     0.823      1077

    accuracy                          0.786      1553
   macro avg      0.780     0.829     0.776      1553
weighted avg      0.852     0.786     0.794      1553



### 示例代码5.69 基于M3E特征的酒店评论情感识别LRC训练与评估

In [74]:
clf = LogisticRegression(max_iter=1000, C=0.01)
clf.fit(x_train, y_train)
pred = clf.predict(x_test)
report = classification_report(y_test, pred, digits=3)
print(report)

              precision    recall  f1-score   support

           0      0.870     0.813     0.840       476
           1      0.920     0.946     0.933      1077

    accuracy                          0.905      1553
   macro avg      0.895     0.880     0.887      1553
weighted avg      0.904     0.905     0.904      1553



## 基于CLIP模型的人像性别分类
### 示例代码5.70 从本地加载CLIP模型

In [75]:
model = sentrans.SentenceTransformer('./fm/clip-ViT-B-32')
# 打印模型的基本信息
clip = model[0].model
p = sum(p.numel() for p in model.parameters())
vocab_size = clip.text_model.embeddings.token_embedding.num_embeddings
max_len = clip.text_model.embeddings.position_embedding.num_embeddings
d = clip.visual_projection.out_features
print("总参数量", f"{p/1e6:.1f} M") # 总参数量 151.3 M
print("词典大小", vocab_size) # 词典大小 49408
print("嵌入维度", d) # 嵌入维度 512
print("输入文本最大长度", max_len) # 输入文本最大长度 77

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


总参数量 151.3 M
词典大小 49408
嵌入维度 512
输入文本最大长度 77


### 示例代码5.71 提取性别标签的CLIP嵌入特征

In [76]:
label_txt = ['a female portrait', 'a male portrait']
label_emb = model.encode(label_txt)

### 示例代码5.72 提取学生肖像的CLIP嵌入特征
默认`pybood-data`在当前目录下

In [78]:
from PIL import Image

columns = '姓名 性别 出生日期 学号 所属学院 入学年份 籍贯 GPA 总学分'.split()
students = pd.read_csv('pybook-data/ch3/rawdata.txt', header=None, names=columns)
s_ids = students['学号'].values
img_path_list = [f'pybook-data/ch3/portraits/{x}.png' for x in s_ids]

# 为每张图像逐一提取CLIP嵌入特征
img_emb = [None]*len(students)
for i, img_path in enumerate(img_path_list):
    img = Image.open(img_path_list[i])
    img = img.resize((224, 224))
    img_emb[i] = model.encode(img)
img_emb = np.array(img_emb)
print(f'图像嵌入特征尺寸: {img_emb.shape}') # 20*512

图像嵌入特征尺寸: (20, 512)


In [79]:
students

,姓名,性别,出生日期,学号,所属学院,入学年份,籍贯,GPA,总学分
0,张伟,男,1999-05-12,2018010101,计算机学院,2018,江苏省南京市,3.6,142
1,李静,女,2000-08-23,2019020203,经济学院,2019,浙江省杭州市,3.8,126
2,王明,男,2001-02-14,2020030305,外国语学院,2020,广东省广州市,3.2,108
3,刘芳,女,1999-11-05,2018040407,法学院,2018,湖北省武汉市,3.7,156
4,陈浩,男,2000-07-19,2019050509,医学院,2019,四川省成都市,3.9,132
5,杨雪,女,2001-04-30,2020060611,艺术学院,2020,陕西省西安市,3.4,112
6,赵阳,男,1999-09-17,2018070713,机械工程学院,2018,山东省济南市,3.5,148
7,周琳,女,2000-12-08,2019080815,数学学院,2019,湖南省长沙市,3.8,128
8,吴强,男,2001-03-21,2020090917,物理学院,2020,河南省郑州市,3.1,98
9,郑婷婷,女,1999-06-25,2018101019,化学学院,2018,福建省福州市,3.6,152


### 示例代码5.73 基于图文嵌入特征相似度进行性别分类

In [80]:
scores = sentrans.util.cos_sim(img_emb, label_emb)
pred = np.argmax(scores, axis=1)
pred = pred.cpu().numpy()
new_table = students[['姓名', '性别']].copy()
new_table['女性相似度'] = [f"{v:.2f}" for v in scores[:, 0]]
new_table['男性相似度'] = [f"{v:.2f}" for v in scores[:, 1]]
new_table['预测标签'] = pred
new_table.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
姓名,张伟,李静,王明,刘芳,陈浩,杨雪,赵阳,周琳,吴强,郑婷婷,孙宇航,朱雨欣,马超,胡佳怡,林志强,徐雅雯,黄博文,高思琪,梁家辉,谢欣怡
性别,男,女,男,女,男,女,男,女,男,女,男,女,男,女,男,女,男,女,男,女
女性相似度,0.20,0.23,0.22,0.23,0.21,0.25,0.21,0.24,0.22,0.24,0.22,0.24,0.20,0.25,0.21,0.25,0.22,0.24,0.21,0.25
男性相似度,0.24,0.17,0.25,0.18,0.25,0.20,0.25,0.19,0.25,0.19,0.25,0.19,0.24,0.20,0.24,0.20,0.25,0.19,0.24,0.21
预测标签,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0
